# Week 25, Day 2: Project: core logic with tests

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w025d2.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


These are the files this page's programs use.

In [ ]:
# Run this cell first: it creates the files used below.
with open('money.py', 'w', encoding='utf-8') as f:
    f.write('"""Reading and showing money and categories."""\nimport re\n\nAMOUNT = re.compile(r"£?\\s*(\\d{1,3}(?:,\\d{3})+|\\d+)(?:\\.(\\d{1,2}))?")\n\n\ndef parse_amount(text):\n    """Text such as \'£1,250.50\' as whole pence. Raises ValueError with a readable message."""\n    text = text.strip()\n    if not text.lstrip("£").strip():\n        raise ValueError("missing amount")\n    m = AMOUNT.fullmatch(text)\n    if m is None:\n        raise ValueError(f"not an amount: {text!r}")\n    total = int(m.group(1).replace(",", "")) * 100 + int((m.group(2) or "").ljust(2, "0"))\n    if total == 0:\n        raise ValueError("amount must be more than £0")\n    return total\n\n\ndef format_pence(pence):\n    """1250 as \'£12.50\', -350 as \'-£3.50\'."""\n    sign = "-" if pence < 0 else ""\n    pounds, rest = divmod(abs(pence), 100)\n    return f"{sign}£{pounds:,}.{rest:02d}"\n\n\ndef clean_category(text):\n    """\' Eating   Out \' as \'eating out\'. Raises ValueError if it isn\'t words of letters."""\n    shown = text.strip()\n    if not shown:\n        raise ValueError("missing category")\n    cleaned = " ".join(shown.lower().split())\n    if len(cleaned) > 20 or not re.fullmatch(r"[a-z]+( [a-z]+)*", cleaned):\n        raise ValueError(f"bad category: {shown!r}")\n    return cleaned\n')
with open('dates.py', 'w', encoding='utf-8') as f:
    f.write('"""Reading dates that people type."""\nfrom datetime import datetime, timedelta\n\nFORMATS = ["%Y-%m-%d", "%d/%m/%Y"]\n\n\ndef parse_date(text, today):\n    """\'2026-11-02\', \'02/11/2026\', \'today\', \'yesterday\' or \'\' as ISO text. No future dates."""\n    shown = text.strip()\n    word = shown.lower()\n    if word in ("", "today"):\n        when = today\n    elif word == "yesterday":\n        when = today - timedelta(days=1)\n    else:\n        when = None\n        for fmt in FORMATS:\n            try:\n                when = datetime.strptime(word, fmt).date()\n                break\n            except ValueError:\n                pass\n        if when is None:\n            raise ValueError(f"not a date: {shown!r}")\n    if when > today:\n        raise ValueError(f"date is in the future: {when.isoformat()}")\n    return when.isoformat()\n')
with open('core.py', 'w', encoding='utf-8') as f:
    f.write('"""The expense tracker\'s data and the operations on it."""\nfrom money import parse_amount, clean_category\nfrom dates import parse_date\n\n\ndef new_data():\n    return {"expenses": [], "budgets": {}, "next_id": 1}\n\n\ndef add_expense(data, amount_text, category_text, note, date_text, today):\n    """Check every field, then add the expense, or raise one ValueError listing every problem."""\n    problems = []\n    amount = category = when = None\n    try:\n        amount = parse_amount(amount_text)\n    except ValueError as e:\n        problems.append(str(e))\n    try:\n        category = clean_category(category_text)\n    except ValueError as e:\n        problems.append(str(e))\n    try:\n        when = parse_date(date_text, today)\n    except ValueError as e:\n        problems.append(str(e))\n    note = note.strip()\n    if len(note) > 60:\n        problems.append("note is longer than 60 characters")\n    if problems:\n        raise ValueError("; ".join(problems))\n    expense = {"id": data["next_id"], "date": when, "amount": amount, "category": category, "note": note}\n    data["expenses"].append(expense)\n    data["next_id"] += 1\n    return expense\n\n\ndef remove_expense(data, expense_id):\n    for i, e in enumerate(data["expenses"]):\n        if e["id"] == expense_id:\n            return data["expenses"].pop(i)\n    raise KeyError(f"no expense #{expense_id}")\n')
with open('sample.py', 'w', encoding='utf-8') as f:
    f.write('"""Two months of made-up spending, for trying things out."""\n\n\ndef sample_data():\n    rows = [\n        (1, "2026-10-01", 320000, "rent", ""),\n        (2, "2026-10-03", 4520, "food", "big shop"),\n        (3, "2026-10-15", 2300, "travel", "train"),\n        (4, "2026-10-28", 1250, "food", "lunch"),\n        (5, "2026-11-01", 320000, "rent", ""),\n        (6, "2026-11-01", 380, "food", "coffee"),\n        (7, "2026-11-02", 6210, "food", "big shop"),\n        (8, "2026-11-02", 2850, "eating out", "Tola\'s birthday"),\n        (9, "2026-11-02", 420, "travel", "bus"),\n    ]\n    return {\n        "expenses": [dict(zip(["id", "date", "amount", "category", "note"], r)) for r in rows],\n        "budgets": {"food": 6000, "travel": 5000, "fun": 3000},\n        "next_id": 11,\n    }\n')

### Exercise: Step 7: read a month

Write `parse_month(text)` that accepts `2026-11` or `11/2026` (the month may be one digit: `1/2027`), after stripping, and returns `"2026-11"`. The month must be 1 to 12. Otherwise raise `ValueError("not a month: '13/2026'")`, with the stripped text.

In [ ]:
import re

def parse_month(text):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def message(text):\n    try:\n        parse_month(text)\n    except ValueError as e:\n        return str(e)\n    raise AssertionError(f\"parse_month({text!r}) should raise ValueError\")\ndef test_good():\n    \"good months\"\n    assert parse_month(\"2026-11\") == \"2026-11\" and parse_month(\" 11/2026 \") == \"2026-11\"\n    assert parse_month(\"1/2027\") == \"2027-01\" and parse_month(\"2026-01\") == \"2026-01\" and parse_month(\"12/2026\") == \"2026-12\"\ndef test_bad():\n    \"bad months\"\n    for t in (\"13/2026\", \"2026-13\", \"2026-00\", \"0/2026\", \"2026-1\", \"Nov 2026\", \"2026/11\", \"\", \"11-2026\", \"123/2026\"):\n        assert message(t) == f\"not a month: {t.strip()!r}\", t")

### Exercise: Step 8: filter and sort

Write `filter_expenses(data, month=None, category=None)` returning a new list of the expenses in that month (`"2026-11"`) and/or category (already cleaned), or all of them if both are `None`. Newest first: by date, latest first, and for the same date, the higher id first. `data["expenses"]` must not change.

In [ ]:
from sample import sample_data

def filter_expenses(data, month=None, category=None):
    ___

data = sample_data()
for e in filter_expenses(data, month="2026-11"):
    print(e["id"], e["date"], e["category"], e["amount"])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("from sample import sample_data as _s\ndef ids(xs):\n    return [e[\"id\"] for e in xs]\ndef test_all():\n    \"everything, newest first\"\n    assert ids(filter_expenses(_s())) == [9, 8, 7, 6, 5, 4, 3, 2, 1]\ndef test_month_category():\n    \"by month and category\"\n    d = _s()\n    assert ids(filter_expenses(d, month=\"2026-10\")) == [4, 3, 2, 1]\n    assert ids(filter_expenses(d, category=\"food\")) == [7, 6, 4, 2]\n    assert ids(filter_expenses(d, month=\"2026-11\", category=\"food\")) == [7, 6]\n    assert filter_expenses(d, month=\"2026-12\") == [] and filter_expenses(d, category=\"fun\") == []\ndef test_order_and_unchanged():\n    \"same date sorts by id, and the data isn't changed\"\n    d = _s()\n    d[\"expenses\"].append({\"id\": 12, \"date\": \"2026-10-03\", \"amount\": 100, \"category\": \"food\", \"note\": \"\"})\n    before = [dict(e) for e in d[\"expenses\"]]\n    assert ids(filter_expenses(d, month=\"2026-10\")) == [4, 3, 12, 2, 1]\n    assert d[\"expenses\"] == before")

### Exercise: Step 9: totals

Write `total(expenses)`, the sum of the amounts in a list of expenses, and `by_category(expenses)`, returning a list of `(category, pence)` pairs, the largest total first, and alphabetical for equal totals.

In [ ]:
from sample import sample_data

def total(expenses):
    ___

def by_category(expenses):
    ___

print(by_category(sample_data()["expenses"]))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("from sample import sample_data as _s\ndef test_total():\n    \"total\"\n    assert total(_s()[\"expenses\"]) == 657930 and total([]) == 0\ndef test_by_category():\n    \"by_category\"\n    got = by_category(_s()[\"expenses\"])\n    assert got == [(\"rent\", 640000), (\"food\", 12360), (\"eating out\", 2850), (\"travel\", 2720)], got\n    assert by_category([]) == []\ndef test_ties():\n    \"equal totals in alphabetical order\"\n    xs = [{\"amount\": 500, \"category\": c} for c in (\"travel\", \"books\", \"fun\")] + [{\"amount\": 700, \"category\": \"zoo\"}]\n    assert by_category(xs) == [(\"zoo\", 700), (\"books\", 500), (\"fun\", 500), (\"travel\", 500)]")

### Exercise: Step 10: set and remove budgets

Write `set_budget(data, category_text, amount_text)`: clean both with `clean_category` and `parse_amount`, collecting problems like `add_expense` (category first, joined by `"; "`, nothing changed), then store the budget and return `(category, pence)`. Setting a budget again replaces it. And `remove_budget(data, category_text)`: clean the category, remove its budget and return the amount, or raise `KeyError("no budget for 'fun'")` (with the cleaned name).

In [ ]:
from money import parse_amount, clean_category
from sample import sample_data

def set_budget(data, category_text, amount_text):
    ___

def remove_budget(data, category_text):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("from sample import sample_data as _s\ndef test_set():\n    \"set and replace\"\n    d = _s()\n    assert set_budget(d, \" Eating Out \", \"\u00a345\") == (\"eating out\", 4500)\n    assert d[\"budgets\"][\"eating out\"] == 4500\n    assert set_budget(d, \"FOOD\", \"75.50\") == (\"food\", 7550) and d[\"budgets\"][\"food\"] == 7550\ndef test_problems():\n    \"problems listed, nothing changed\"\n    d = _s()\n    try:\n        set_budget(d, \"Food & Drink\", \"-5\")\n        assert False, \"should raise ValueError\"\n    except ValueError as e:\n        assert str(e) == \"bad category: 'Food & Drink'; not an amount: '-5'\", str(e)\n    try:\n        set_budget(d, \"food\", \"0\")\n        assert False, \"should raise ValueError\"\n    except ValueError as e:\n        assert str(e) == \"amount must be more than \u00a30\"\n    assert d == _s()\ndef test_remove():\n    \"remove\"\n    d = _s()\n    assert remove_budget(d, \" Fun \") == 3000 and \"fun\" not in d[\"budgets\"]\n    try:\n        remove_budget(d, \"fun\")\n        assert False, \"should raise KeyError\"\n    except KeyError as e:\n        assert e.args[0] == \"no budget for 'fun'\"")

### Exercise: Step 11: summarise a month

Write `month_summary(data, month)` returning a dictionary with:

- `"month"`: the month;
- `"count"` and `"total"`: how many expenses that month, and their total;
- `"rows"`: one dictionary per category that has spending that month **or** a budget, with keys `"category"`, `"spent"` (0 if none), `"budget"` (`None` if none) and `"left"` (budget minus spent, or `None` with no budget); most spent first, alphabetical for ties;
- `"over"`: the categories whose `"left"` is below 0, in the same order.

Use `filter_expenses` and `by_category`, given in the starter.

In [ ]:
from sample import sample_data

def filter_expenses(data, month=None, category=None):
    chosen = [e for e in data["expenses"]
              if (month is None or e["date"][:7] == month)
              and (category is None or e["category"] == category)]
    return sorted(chosen, key=lambda e: (e["date"], e["id"]), reverse=True)

def total(expenses):
    return sum(e["amount"] for e in expenses)

def by_category(expenses):
    totals = {}
    for e in expenses:
        totals[e["category"]] = totals.get(e["category"], 0) + e["amount"]
    return sorted(totals.items(), key=lambda pair: (-pair[1], pair[0]))

def month_summary(data, month):
    ___

for row in month_summary(sample_data(), "2026-11")["rows"]:
    print(row)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("from sample import sample_data as _s\ndef test_november():\n    \"November\"\n    s = month_summary(_s(), \"2026-11\")\n    assert s[\"month\"] == \"2026-11\" and s[\"count\"] == 5 and s[\"total\"] == 329860\n    assert s[\"rows\"] == [\n        {\"category\": \"rent\", \"spent\": 320000, \"budget\": None, \"left\": None},\n        {\"category\": \"food\", \"spent\": 6590, \"budget\": 6000, \"left\": -590},\n        {\"category\": \"eating out\", \"spent\": 2850, \"budget\": None, \"left\": None},\n        {\"category\": \"travel\", \"spent\": 420, \"budget\": 5000, \"left\": 4580},\n        {\"category\": \"fun\", \"spent\": 0, \"budget\": 3000, \"left\": 3000},\n    ], s[\"rows\"]\n    assert s[\"over\"] == [\"food\"]\ndef test_october_and_empty():\n    \"October, and a month with nothing\"\n    s = month_summary(_s(), \"2026-10\")\n    assert s[\"count\"] == 4 and s[\"over\"] == [] and [r[\"category\"] for r in s[\"rows\"]] == [\"rent\", \"food\", \"travel\", \"fun\"]\n    assert s[\"rows\"][2] == {\"category\": \"travel\", \"spent\": 2300, \"budget\": 5000, \"left\": 2700}\n    e = month_summary(_s(), \"2027-01\")\n    assert e[\"count\"] == 0 and e[\"total\"] == 0 and e[\"over\"] == []\n    assert [r[\"spent\"] for r in e[\"rows\"]] == [0, 0, 0] and [r[\"category\"] for r in e[\"rows\"]] == [\"food\", \"fun\", \"travel\"]\ndef test_exactly_on_budget():\n    \"spending exactly the budget isn't over\"\n    d = _s()\n    d[\"budgets\"][\"food\"] = 6590\n    assert month_summary(d, \"2026-11\")[\"over\"] == []")

### Exercise: Step 12: monthly totals

Write `next_month(month)` (`"2026-12"` gives `"2027-01"`) and `monthly_totals(data)`: a list of `(month, pence)` pairs from the earliest month with spending to the latest, in order, including months in between with a total of 0. No expenses at all gives `[]`.

In [ ]:
from sample import sample_data

def next_month(month):
    ___

def monthly_totals(data):
    ___

print(monthly_totals(sample_data()))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("from sample import sample_data as _s\ndef test_next():\n    \"next_month\"\n    assert next_month(\"2026-11\") == \"2026-12\" and next_month(\"2026-12\") == \"2027-01\" and next_month(\"2027-09\") == \"2027-10\"\ndef test_sample():\n    \"the sample\"\n    assert monthly_totals(_s()) == [(\"2026-10\", 328070), (\"2026-11\", 329860)]\ndef test_gaps():\n    \"gaps are filled with 0\"\n    d = {\"expenses\": [{\"id\": 1, \"date\": \"2027-02-14\", \"amount\": 999, \"category\": \"fun\", \"note\": \"\"},\n                      {\"id\": 2, \"date\": \"2026-11-30\", \"amount\": 100, \"category\": \"fun\", \"note\": \"\"},\n                      {\"id\": 3, \"date\": \"2026-11-01\", \"amount\": 50, \"category\": \"food\", \"note\": \"\"}],\n         \"budgets\": {}, \"next_id\": 4}\n    assert monthly_totals(d) == [(\"2026-11\", 150), (\"2026-12\", 0), (\"2027-01\", 0), (\"2027-02\", 999)]\n    assert monthly_totals({\"expenses\": [], \"budgets\": {}, \"next_id\": 1}) == []")

### Exercise: Step 13: test the core

Write a pytest suite for the core in `core.py` (this exercise's version includes everything from today). Use a fixture that builds a small data dictionary of your own, at least one parametrized test, and `pytest.raises`. It must pass on the real code and catch all **eight** broken versions.

In [ ]:
# Run this cell first: it creates the files used below.
with open('core.py', 'w', encoding='utf-8') as f:
    f.write('"""The expense tracker\'s data and the operations on it."""\nfrom money import parse_amount, clean_category\nfrom dates import parse_date\n\n\ndef new_data():\n    return {"expenses": [], "budgets": {}, "next_id": 1}\n\n\ndef add_expense(data, amount_text, category_text, note, date_text, today):\n    """Check every field, then add the expense, or raise one ValueError listing every problem."""\n    problems = []\n    amount = category = when = None\n    try:\n        amount = parse_amount(amount_text)\n    except ValueError as e:\n        problems.append(str(e))\n    try:\n        category = clean_category(category_text)\n    except ValueError as e:\n        problems.append(str(e))\n    try:\n        when = parse_date(date_text, today)\n    except ValueError as e:\n        problems.append(str(e))\n    note = note.strip()\n    if len(note) > 60:\n        problems.append("note is longer than 60 characters")\n    if problems:\n        raise ValueError("; ".join(problems))\n    expense = {"id": data["next_id"], "date": when, "amount": amount, "category": category, "note": note}\n    data["expenses"].append(expense)\n    data["next_id"] += 1\n    return expense\n\n\ndef remove_expense(data, expense_id):\n    """Remove and return the expense with this id. KeyError if there isn\'t one."""\n    for i, e in enumerate(data["expenses"]):\n        if e["id"] == expense_id:\n            return data["expenses"].pop(i)\n    raise KeyError(f"no expense #{expense_id}")\n\n\ndef filter_expenses(data, month=None, category=None):\n    """A new list of the matching expenses, newest first (by date, then id)."""\n    chosen = [e for e in data["expenses"]\n              if (month is None or e["date"][:7] == month)\n              and (category is None or e["category"] == category)]\n    return sorted(chosen, key=lambda e: (e["date"], e["id"]), reverse=True)\n\n\ndef total(expenses):\n    return sum(e["amount"] for e in expenses)\n\n\ndef by_category(expenses):\n    """(category, pence) pairs, largest first, alphabetical for ties."""\n    totals = {}\n    for e in expenses:\n        totals[e["category"]] = totals.get(e["category"], 0) + e["amount"]\n    return sorted(totals.items(), key=lambda pair: (-pair[1], pair[0]))\n\n\ndef month_summary(data, month):\n    """Count, total, a row per category with spending or a budget, and the categories over budget."""\n    expenses = filter_expenses(data, month=month)\n    spent = dict(by_category(expenses))\n    rows = []\n    for category in set(spent) | set(data["budgets"]):\n        budget = data["budgets"].get(category)\n        used = spent.get(category, 0)\n        rows.append({"category": category, "spent": used, "budget": budget,\n                     "left": None if budget is None else budget - used})\n    rows.sort(key=lambda r: (-r["spent"], r["category"]))\n    return {"month": month, "count": len(expenses), "total": total(expenses), "rows": rows,\n            "over": [r["category"] for r in rows if r["left"] is not None and r["left"] < 0]}\n')

In [ ]:
import pytest
from datetime import date
from core import (new_data, add_expense, remove_expense, filter_expenses,
                  total, by_category, month_summary)

TODAY = date(2026, 11, 2)

@pytest.fixture
def data():
    d = new_data()
    ___
    return d

In [ ]:
# Check: run the exercise cell above first, then this one.
check("GOOD = open(\"core.py\").read()\nBROKEN = {\n    \"half-added on error\": GOOD.replace('''    note = note.strip()\n    if len(note) > 60:''', '''    if amount is not None and category is not None and when is not None:\n        data[\"expenses\"].append({\"id\": data[\"next_id\"], \"date\": when, \"amount\": amount, \"category\": category, \"note\": note})\n    note = note.strip()\n    if len(note) > 60:'''),\n    \"ids reused\": GOOD.replace('            return data[\"expenses\"].pop(i)', '            data[\"next_id\"] -= 1\\n            return data[\"expenses\"].pop(i)'),\n    \"remove ignores missing\": GOOD.replace('    raise KeyError(f\"no expense #{expense_id}\")', \"    return None\"),\n    \"oldest first\": GOOD.replace(\"key=lambda e: (e[\\\"date\\\"], e[\\\"id\\\"]), reverse=True)\", \"key=lambda e: (e[\\\"date\\\"], e[\\\"id\\\"]))\"),\n    \"category ignored\": GOOD.replace('and (category is None or e[\"category\"] == category)', \"\"),\n    \"ties not alphabetical\": GOOD.replace(\"key=lambda pair: (-pair[1], pair[0])\", \"key=lambda pair: -pair[1]\"),\n    \"budgets without spending hidden\": GOOD.replace(\"for category in set(spent) | set(data[\\\"budgets\\\"]):\", \"for category in set(spent):\"),\n    \"exactly on budget is over\": GOOD.replace('r[\"left\"] < 0]}', 'r[\"left\"] <= 0]}'),\n}\ndef test_style():\n    \"a fixture, parametrize and pytest.raises\"\n    assert \"@pytest.fixture\" in source and \"parametrize\" in source and \"pytest.raises\" in source\ndef test_catches():\n    \"your tests pass on the real code and catch all eight broken versions\"\n    results = against(source, \"core\", {\"correct\": GOOD, **BROKEN})\n    write_file(\"core.py\", GOOD)\n    assert results[\"correct\"] == [], f\"these fail on the real code: {results['correct']}\"\n    missed = [k for k in BROKEN if not results[k]]\n    assert not missed, f\"not caught: {missed}\"")

> **Tests at a seam.** These tests check `core.py` through its functions, the way the interface will call them, not through its internals. Tomorrow you'll change how data is loaded and saved, and on Day 5 you may tidy `core.py` itself. As long as these tests stay green, nothing a user depends on has changed. That's what makes the rest of the week safe.

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

## Where we are

Yesterday's code is now split into modules, as planned. They're in the files below (open them to read them): `money.py` has `parse_amount`, `format_pence` and `clean_category`; `dates.py` has `parse_date`; `core.py` has `new_data`, `add_expense` and `remove_expense`. There's also `sample.py`, with `sample_data()` returning two months of made-up expenses to try things on.

Today's functions also belong in `core.py`. In each editor, they're written below a line that imports what's already there.

Notice the **docstrings**: each function in a module says, in a line or two, what it takes and gives back. When the modules grow, that's what you'll read instead of the code.

Number 10 is missing from the sample: it was added and then removed, so `next_id` is 11.

## Step 7: months

Lists and summaries work by month, written `"2026-11"`, the first 7 characters of an ISO date. People will type months in two ways, so they need reading and checking like everything else.

## Step 8: filtering

The `list` command shows expenses, optionally for one month or one category, newest first. Filtering returns a **new** list; the stored data mustn't change order just because someone looked at it.

## Step 9: totals by category

## Step 10: budgets

A budget is a monthly limit for one category, stored in `data["budgets"]` as pence. It reuses yesterday's checks, so `budget Food £60` works just like adding an expense.

## Step 11: a month's summary

The `summary` command is the heart of the tool. Its function gathers everything the interface will print, as data, and the interface only has to lay it out.

## Step 12: month by month

A trend needs **every** month in the range, including any with no spending at all. A month with nothing isn't missing data; it's a month where you spent £0, and leaving it out would make a chart lie.

## Step 13: a test suite for the core

The core is done for now, and the next three days will build on top of it, so this is the moment to pin its behaviour down with tests. Here's today's work collected into `core.py`, with docstrings; your job is to write `tests/test_core.py` for it.

## Quiz

## Before you go

Add today's functions to `core.py` in your own `spend` folder, with docstrings, and today's tests to `tests/test_core.py`. Run `pytest` and commit when it's green. Tomorrow: saving the data safely to a file, and importing a bank statement.